### What changes during training?
- Weights and biases (learnable parameters) are updated during training to reduce the loss.

### What stays fixed during training?
- The training dataset, model architecture, and loss function usually stay fixed during training.

### Why are weights called parameters?
- Weights are called parameters because they are values learned from data and determine how the model transforms inputs into outputs.
- Parameters are the learnable values that define the behavior of the model.

## Create Tiny Dataset

In [1]:
import torch

X = torch.tensor([[1.0],
                  [2.0],
                  [3.0],
                  [4.0]])

y = torch.tensor([[3.0],
                  [5.0],
                  [7.0],
                  [9.0]])

### Why is shape (4,1) instead of (4,)?
- Because we have created dataset which has 4 samples and 1 feature thus it has shape (4,1) representing that. while (4,) is a one dimensional vector and it can mean two things either it is a tensor with 1 sample and 4 features or 4 sample and 1 feature to avoid this misinterpretation dataset is created in (4,1) shape. 

### What does each row represent?
- Each row represents sample.

## Create Model

In [2]:
model = torch.nn.Linear(1, 1)

print(model.weight)
print(model.bias)

Parameter containing:
tensor([[0.4013]], requires_grad=True)
Parameter containing:
tensor([-0.2918], requires_grad=True)


### What values were initialized?
- Values of weight and bias is initialized randomly between 0 and 1.

### Why aren't they initialized to zero?
- If weights are initialized to the same value (such as zero or one), all neurons in a layer will compute identical outputs and receive identical gradient updates, effectively reducing the network's capacity to learn complex patterns. 

## Forward Pass

In [3]:
preds = model(X)
preds

tensor([[0.1096],
        [0.5109],
        [0.9123],
        [1.3136]], grad_fn=<AddmmBackward0>)

### Are predictions close to actual targets? and Why not?
- No, because weight and bias are random and model is not yet trained or parameters are not have been updated yet in the direction which minimizes the loss.

## Loss Function

In [4]:
loss_fn = torch.nn.MSELoss()
loss = loss_fn(preds, y)
loss

tensor(31.1617, grad_fn=<MseLossBackward0>)

### What does loss measure?
- It measures how far the prediction are from the expected values.

### Why is loss a scalar?
- Loss is a scalar because optimization requires a single objective value that summarizes overall model error. Gradients are computed with respect to this single objective and used to update parameters.

## Backpropagation

In [5]:
loss.backward()

print(model.weight.grad)
print(model.bias.grad)

tensor([[-30.4386]])
tensor([-10.5768])


### What do these gradients represent?
- This gradient represents how much changing those parameters will affect the loss. If the weight increases slightly,
how will the loss change?

### Why are gradients attached to parameters?
- Gradients are attached to parameters because parameters are the learnable components of the model. The optimizer uses their gradients to determine how they should be updated to reduce loss.

## Optimizer

In [6]:
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)

### What is learning rate?
- Learning rate determines how far the optimizer moves in the direction suggested by the gradient.  

### What happens if it is too small or too large?
- If learning rate is too small optimizer would make small steps towards the direction which minimizes the loss but since it is too small we might have to take many steps before reaching the local minima thus training becomes very slow
- If it is too large it will take big steps but it might overshoot (skip the local minima completely).

## Parameter Update

In [7]:
optimizer.step()

In [8]:
print(model.weight)
print(model.bias)

Parameter containing:
tensor([[0.7057]], requires_grad=True)
Parameter containing:
tensor([-0.1860], requires_grad=True)


- At the time of initialization weight was different and bias was different but after one step of optimization value has changed.
- new_parameter = old_parameter - (learning_rate * gradient_of_parameter)

### What changed?
- The values of the weight and bias changed because the optimizer used their gradients to update them in a direction that reduces loss.

### Did weights update automatically?
- Weight or any parameter is updated only after calling `optimizer.step()`.
- First we need to calculate gradient using `loss.backward()` then parameter is updated using `optimizer.step()` then to prepare for next iteration we clear old gradient using `optimizer.zero_grad()`.

## Gradient Reset

In [9]:
optimizer.zero_grad()

### Why is this required every iteration and What happens if omitted?
- optimizer.zero_grad() is required because PyTorch accumulates gradients by default. Without clearing them, gradients from previous iterations would be added to current gradients, causing incorrect parameter updates.

## One Full Training Step

1. forward pass
2. loss calculation
3. backward pass
4. optimizer step
5. zero gradients

## Full Training Loop

In [10]:
for epoch in range(100):
    
    pred = model(X)
    
    loss = loss_fn(pred, y)
    
    optimizer.zero_grad()
    
    loss.backward()
    
    optimizer.step()
    
    if epoch % 10 == 0:
        print("Loss :", loss.item())
        print("Weight :", model.weight.item())
        print("Bias :", model.bias.item())
        print()    

Loss : 21.645008087158203
Weight : 0.9591744542121887
Bias : -0.09756641089916229

Loss : 0.6283613443374634
Weight : 2.0161056518554688
Bias : 0.28388288617134094

Loss : 0.08063220977783203
Weight : 2.1806907653808594
Bias : 0.361176997423172

Loss : 0.06271414458751678
Weight : 2.2019031047821045
Bias : 0.3890773355960846

Loss : 0.05872170999646187
Weight : 2.200209856033325
Bias : 0.4085758328437805

Loss : 0.055294882506132126
Weight : 2.1949825286865234
Bias : 0.4262794852256775

Loss : 0.052076295018196106
Weight : 2.189333438873291
Bias : 0.4432641267776489

Loss : 0.04904511198401451
Weight : 2.1837587356567383
Bias : 0.4597155451774597

Loss : 0.04619041830301285
Weight : 2.1783335208892822
Bias : 0.4756758511066437

Loss : 0.04350198060274124
Weight : 2.1730661392211914
Bias : 0.491163969039917



### Why is order important?
- We have to follow steps in order to train a model else training will not happen. Order goes like this first we make a prediction on input data then we calculate loss (difference between predicted and actual values but we also scale it to single number for whole model) next step is to calculate gradient of parameter which affect loss (weight and bias) now we update the parameters to minimize the loss according to the gradient at last we clear the gradient and set it to zero but in practice we do it before calculating gradient so that this iteration is not affected by gradient of previous iteration.
z
### What happens if backward is before loss? 
- Their will be no loss to calculate gradient on. `backward()` needs a scalar objective (loss) to differentiate.

### What happens if step before backward?
- parameter.grad is either None or contains stale gradients from a previous iteration. So `optimizer.step()` has no meaningful information to use.

## Monitor Learning

### Is loss decreasing? 
- Yes

### What does decreasing loss imply?
- Decreasing loss indicates that the model is fitting the training data better by adjusting its parameters. However, decreasing training loss alone does not guarantee good generalization to unseen data.

## Final Parameters

In [11]:
print(model.weight)
print(model.bias)

Parameter containing:
tensor([[2.1685]], requires_grad=True)
Parameter containing:
tensor([0.5047], requires_grad=True)


### What values do they converge toward?
- Weight converge towards 2 and bias converges towards 1.

### Compare with actual equation:
- As out X is 1.0, 2.0, 3.0, 4.0 and their corresponding y is 3.0, 5.0, 7.0, 9.0 so equation becomes $y = 2x + 1$. And which is what our model is converging to.

## Prediction After Training

In [12]:
model(torch.tensor([5.0]))

tensor([11.3470], grad_fn=<ViewBackward0>)

### Is prediction close to expected value?
- Yes prediction is close to 11 which is expected value as our equation was $y = 2x + 1$

### Why can the model generalize to unseen input?
- Because it has learned patter of input and output or you can say it has learned the relation between input and output value. 

## Complete Mental Model

Initial random weights -> Forward pass -> Loss -> Gradients -> Weight update -> Better predictions